# 3장 — SFT: 시범 보여주기 ⚙️ (핵심)

웹 챕터의 한 문장 **"SFT = 모범답안에 대한 크로스 엔트로피 최소화"**를 코드로 증명합니다.

2장의 초소형 모델(빈도표)을 가져와서:
1. 모범답안을 토큰별로 **채점**(−log₂ q, 5부 4장의 자)
2. **훈련 스텝**(표 갱신 = 가중치 갱신)을 반복하며 손실이 내려가는 **학습 곡선**을 직접 그립니다
3. **손실 마스킹**(지시는 채점 제외)까지

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch3')

## 1. 미니 모델 + 채점기 만들기

모델 = "앞 단어 → 다음 단어 확률" 표(2장 그대로).
채점기 = 모범답안의 각 토큰에 모델이 준 확률 q 를 찾아 **−log₂ q** 를 매기는 함수.
본 적 없는 조합에도 확률 0 대신 아주 작은 확률을 주도록 **스무딩**을 넣습니다
(q=0 이면 손실이 무한대, 5부 4장에서 본 함정).

In [ ]:
import math
from collections import Counter, defaultdict

# 사전학습을 마친 상태의 미니 모델 (인터넷 = 일상 글 — 답변 형식은 거의 본 적 없음)
PRETRAIN = [
    '내일 날씨가 좋다고 한다',
    '내일 회의 자료를 준비해야 한다',
    '회의를 마치고 점심을 먹었다',
    '취소해 달라는 요청이 많았다',
]

counts = defaultdict(Counter)
vocab = set()
for line in PRETRAIN:
    words = line.split()
    vocab.update(words)
    for a, b in zip(words, words[1:]):
        counts[a][b] += 1

SMOOTH = 0.01  # 본 적 없는 조합에도 작은 확률 (q=0 → 손실 무한대 방지)

def q_of(prev, nxt):
    """모델이 '{prev} 다음은 {nxt}' 에 주는 확률 (스무딩 포함)."""
    vocab.add(nxt)
    c = counts[prev]
    return (c[nxt] + SMOOTH) / (sum(c.values()) + SMOOTH * len(vocab))

def token_losses(tokens):
    """토큰열의 (다음 토큰, 손실 −log₂ q) 목록."""
    return [(b, -math.log2(q_of(a, b))) for a, b in zip(tokens, tokens[1:])]

# SFT 시범 1개 (대화 템플릿으로 이어 붙인 한 줄)
DEMO = '요청: 존댓말로 바꿔줘: 내일 회의 취소해 답변: 내일 회의를 취소해 주시겠어요?'.split()
ANSWER_START = DEMO.index('답변:')  # 이 뒤부터가 채점 대상 (손실 마스킹)

print('훈련 전 답변 토큰별 손실 (비트):')
for tok, loss in token_losses(DEMO)[ANSWER_START:]:
    print(f'   {tok:10s} {loss:5.1f}')

손실이 전부 큽니다. 사전학습 데이터에 답변 형식이 없었으니 당연해요.

## 2. 훈련 = 표를 시범 쪽으로 밀기

진짜 SFT 는 경사하강으로 가중치를 밉니다. 우리 미니 모델의 "가중치"는 빈도표이니,
훈련 한 스텝 = **시범에 나온 (앞 단어→다음 단어) 조합의 칸을 +1** 하는 것으로 흉내 냅니다.
이때 **마스킹**을 씁니다. 답변 부분의 조합만 갱신합니다.

In [ ]:
def avg_answer_loss():
    losses = [loss for _, loss in token_losses(DEMO)[ANSWER_START:]]
    return sum(losses) / len(losses)

def train_step():
    """SFT 한 스텝 — 답변 부분(마스킹 통과)의 조합만 표에 +1."""
    for a, b in list(zip(DEMO, DEMO[1:]))[ANSWER_START:]:
        counts[a][b] += 1

curve = [avg_answer_loss()]
for _ in range(12):
    train_step()
    curve.append(avg_answer_loss())

for step, loss in enumerate(curve):
    bar = '█' * int(loss * 4)
    print(f'스텝 {step:2d} | 평균 손실 {loss:5.2f} {bar}')

In [ ]:
# 학습 곡선 — 4부 11장에서 본 loss 곡선의 초소형 재현
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(range(len(curve)), curve, marker='o', color='#6366f1')
plt.xlabel('훈련 스텝')
plt.ylabel('평균 손실 (비트)')
plt.title('미니 SFT 학습 곡선: 시범이 가중치에 새겨지는 중')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 훈련의 결과: 제로샷으로 행동이 나온다

2장의 퓨샷은 예시를 매번 붙여야 했죠. 훈련을 마친 지금, 표 자체가 바뀌었으니
`답변:` 뒤의 분포를 그냥 확인해 봅시다. 그리고 검증까지.

In [ ]:
top = counts['답변:'].most_common(1)[0]
print(f"'답변:' 뒤에 가장 유력한 다음 단어: '{top[0]}' (표에 {top[1]}번 기록됨)")

# 검증 1 — 훈련으로 손실이 크게 내려갔다
assert curve[-1] < curve[0] * 0.3, 'SFT 훈련은 손실을 크게 낮춰야 한다!'
# 검증 2 — 마스킹: 지시 부분 조합('존댓말로' 다음)은 표가 갱신되지 않았다
assert counts['존댓말로']['바꿔줘:'] == 0, '마스킹된 지시 부분은 학습되지 않아야 한다!'
print('\n✔ 확인: 손실 하강(학습됨) + 지시 부분 미학습(마스킹), SFT 의 두 기둥')

## 4. 관찰 정리 + 🤖 AI 연결

- **채점기 = 크로스 엔트로피**(−log₂ q, 5부 4장), **엔진 = 반복 갱신**(4부 11장). 새 수학 없음.
- 훈련 후에는 문맥 조작 없이도(제로샷) 행동이 나온다. 표(가중치) 자체가 바뀌었으니까.
- 마스킹 덕에 지시 부분은 배우지 않는다. 채점되는 토큰이 곧 배우는 대상.
- 진짜와 다른 점: 진짜 SFT 는 표에 +1 이 아니라 **기울기만큼** 밀고(2부 9장),
  시범도 수만 개를 씁니다. 원리는 동일.

**다음 챕터**: 4장, 시범으로 표현 못 하는 "어느 쪽이 더 나은가"를 숫자로 바꾸는 보상 모델.
3부의 조건부 확률과 2부의 시그모이드가 돌아옵니다.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch3')`.